**Probando conexión a Postgres**

In [6]:
from config.database import obtener_conexion

with obtener_conexion() as conn:
    with conn.cursor() as cursor:
        cursor.execute("SELECT version()")

        version = cursor.fetchone()

print(version)

('PostgreSQL 17.5 on x86_64-windows, compiled by msvc-19.44.35209, 64-bit',)


In [ ]:
from config.database import obtener_conexion

query = """

CREATE TABLE clientes (
    id_cliente SERIAL PRIMARY KEY,
    nombre VARCHAR(100) NOT NULL,
    telefono VARCHAR(30),
    email VARCHAR(255)
);

CREATE TABLE proveedores (
    id_proveedor SERIAL PRIMARY KEY,
    nombre VARCHAR(100) NOT NULL,
    telefono VARCHAR(30),
    email VARCHAR(255)
);

CREATE TABLE servicios (
    id_servicio SERIAL PRIMARY KEY,
    nombre_servicio VARCHAR(100) NOT NULL,
    duracion_minutos INTEGER NOT NULL,
    precio NUMERIC(10,2) NOT NULL
);

CREATE TABLE turnos (
    id_turno SERIAL PRIMARY KEY,

    id_cliente INTEGER NOT NULL,
    id_proveedor INTEGER NOT NULL,
    id_servicio INTEGER NOT NULL,

    fecha_turno TIMESTAMP NOT NULL,

    estado VARCHAR(20) NOT NULL DEFAULT 'PENDIENTE',

    notas TEXT,

    fecha_creacion TIMESTAMP DEFAULT CURRENT_TIMESTAMP,

    CONSTRAINT fk_turno_cliente
        FOREIGN KEY (id_cliente)
        REFERENCES clientes(id_cliente),

    CONSTRAINT fk_turno_proveedor
        FOREIGN KEY (id_proveedor)
        REFERENCES proveedores(id_proveedor),

    CONSTRAINT fk_turno_servicio
        FOREIGN KEY (id_servicio)
        REFERENCES servicios(id_servicio),

    CONSTRAINT chk_estado_turno
        CHECK (
            estado IN (
                'PENDIENTE',
                'CONFIRMADO',
                'COMPLETADO',
                'CANCELADO',
                'NO_ASISTIO'
            )
        )
);

CREATE TABLE pagos (
    id_pago SERIAL PRIMARY KEY,

    id_turno INTEGER NOT NULL,

    monto NUMERIC(10,2) NOT NULL,

    metodo_pago VARCHAR(30),

    estado VARCHAR(20) NOT NULL DEFAULT 'PENDIENTE',

    fecha_pago TIMESTAMP,

    fecha_creacion TIMESTAMP DEFAULT CURRENT_TIMESTAMP,

    CONSTRAINT fk_pago_turno
        FOREIGN KEY (id_turno)
        REFERENCES turnos(id_turno),

    CONSTRAINT chk_estado_pago
        CHECK (
            estado IN (
                'PENDIENTE',
                'PAGADO',
                'REEMBOLSADO'
            )
        )
);

        """

with obtener_conexion() as conn:
    with conn.cursor() as cursor:
        cursor.execute(query)

Prueba clientes

In [3]:
from repositories.clientes import crear_cliente
from repositories.clientes import obtener_clientes

crear_cliente(
    "Juan Pérez",
    "2611234567",
    "juan@gmail.com"
)

clientes = obtener_clientes()

print(clientes)

[(1, 'Juan Pérez', '2611234567', 'juan@gmail.com')]


In [5]:
from auth import hashear_password

password = "holaComoEstas"

print(hashear_password(password))

$2b$12$Z7GstwzfFbFTtuCOWcG8zuxgT7y.HWpkWwOnGCVt6yf2Nw.sIwdVG
